# WFD regions, mini-surveys and micro-surveys: cumulative visits versus time, in baseline survey v5.3

This notebook is the counterpart of `04_DDF_NVisitsVsTime` for everything that is **not a DDF**. It shows, for each **WFD region** (low dust, bulge, Virgo, SMC-LMC), for each **mini-survey** (NES, SCP, MW non-WFD, ...) and for each **micro-survey** (DDF, RGES, near-sun twilight, ToO), how the **cumulative number of visits grows with time**, together with the totals of WFD, of the mini-surveys and of the micro-surveys. It also shows the **fraction of the survey visits** spent in each region, as a function of the **total number of visits** of the survey and as a function of time.

- creation date : 2026-10-10
- companion notebooks: `02_SurveyAreas` (area and number of visits of each region, summed over the survey), `04_DDF_NVisitsVsTime` (same study for the DDFs)
- survey modes definition (WFD, mini-surveys, micro-surveys): https://github.com/lsst/survey-strategy_lsst_io

## Overview and outputs

**Survey modes used here**

| Mode | Components | How a visit is attributed |
|---|---|---|
| WFD regions | `low dust` (map labels `lowdust` + `euclid_overlap`), `bulge` (`bulgy`), `virgo`, `SMC-LMC` (`LMC_SMC`), and `Total WFD` | label of the HEALPix pixel containing the visit centre |
| Mini-surveys | `NES` (`nes`), `SCP` (`scp`), `MW non-WFD` (`dusty_plane`), and `Total mini-surveys` | same |
| Micro-surveys | `DDF`, `RGES`, `TwilightNEO`, `ToO`, and `Total micro-surveys` | `scheduler_note` of the visit |

Each visit is counted **once**: micro-survey visits are selected first from `scheduler_note` (priority ToO > TwilightNEO > RGES > DDF), and all the other visits are attributed to a WFD region or a mini-survey through the pixel label of the scheduler target map (`CurrentAreaMap`). The WFD + mini-survey + micro-survey totals therefore add up to the number of visits of the survey (visits falling in a pixel without label are counted in `Unassigned`). In `02_SurveyAreas` the `WFDlabelStacker` is used instead: a visit counts for a region if more than 40% of its field of view lies in it, so a visit can count for several regions and the numbers can differ slightly at the region boundaries.

The third visits of triplets (`Triplet (third)`) are WFD visits, so they are **not** a separate category; they are only drawn as a dashed overlay in the micro-survey figure.

**Steps**

1. Build the region labels from the scheduler target map.
2. Load all the visits of the OpSim database and attribute each of them to one category.
3. Summary table per region and number of visits per survey year.
4. Cumulative number of visits versus time (Figures 1-4).
5. Fraction of the survey visits versus the total number of visits (Figure 5) and versus time (Figure 6).

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) go to `DATA07_WFDMiniMicroS_NVisitsVsTime/`; all figures (`.png` and `.pdf`) go to `FIG07_WFDMiniMicroS_NVisitsVsTime/`.

## 0. Setup

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

from rubin_scheduler.utils import SURVEY_START_MJD
from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
plt.rcParams.update(
    {
        "font.size": 14,  # Taille générale
        "axes.labelsize": 16,  # Labels X et Y
        "axes.titlesize": 18,  # Titre des axes
        "xtick.labelsize": 14,  # Graduations X
        "ytick.labelsize": 14,  # Graduations Y
        "legend.fontsize": 16,  # Texte de la légende
        "legend.title_fontsize": 16,
    }
)

In [ ]:
# Output directories: tables go to DATA_DIR, all figures go to FIG_DIR
DATA_DIR = "DATA07_WFDMiniMicroS_NVisitsVsTime"
FIG_DIR = "FIG07_WFDMiniMicroS_NVisitsVsTime"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by an explicit path to analyse another run.
# opsdb = get_baseline()
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

YEAR = 365.25  # days
BANDS = list("ugrizy")
NSIDE = 64  # HEALPix resolution of the scheduler target map

# Region definition: name -> list of labels of the scheduler target map.
# 'euclid_overlap' is merged with 'lowdust' (as the ExGal region of 02_SurveyAreas).
WFD_REGIONS = {
    "low dust": ["lowdust", "euclid_overlap"],
    "bulge": ["bulgy"],
    "virgo": ["virgo"],
    "SMC-LMC": ["LMC_SMC"],
}
MINI_SURVEYS = {
    "NES": ["nes"],
    "SCP": ["scp"],
    "MW non-WFD": ["dusty_plane"],
}

## 1. Region labels

`CurrentAreaMap` returns one survey-mode label per HEALPix pixel (`labels`). DDFs and the other micro-surveys are added later by the scheduler, so they are not in this map. A label present in the map but not listed in `WFD_REGIONS` or `MINI_SURVEYS` is automatically added as a mini-survey with its own name, so no region is silently forgotten. The area of each region is the number of its pixels times the pixel area (deg2).

In [ ]:
# Scheduler footprint: one survey-mode label per HEALPix pixel
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()
labels = np.asarray(labels).astype(str)
label_names = [l for l in np.unique(labels) if l != ""]
pixarea = hp.nside2pixarea(NSIDE, degrees=True)
print("Labels in the target map:", label_names)

# Labels configured above but absent from the map
for name, labs in {**WFD_REGIONS, **MINI_SURVEYS}.items():
    for l in labs:
        if l not in label_names:
            print(f"WARNING: label {l} of region {name} is not in the target map")

# Labels of the map not assigned to any region: add them as mini-surveys
used_labels = {l for labs in list(WFD_REGIONS.values()) + list(MINI_SURVEYS.values()) for l in labs}
for l in label_names:
    if l not in used_labels:
        print(f"Label {l} is not in WFD_REGIONS or MINI_SURVEYS: added as a mini-survey")
        MINI_SURVEYS[l] = [l]

wfd_names = list(WFD_REGIONS)
mini_names = list(MINI_SURVEYS)
micro_names = ["DDF", "RGES", "TwilightNEO", "ToO"]


def area_of_labels(labs):
    return sum(int((labels == l).sum()) for l in labs) * pixarea


area_of = {n: area_of_labels(labs) for n, labs in {**WFD_REGIONS, **MINI_SURVEYS}.items()}
area_of["Total WFD"] = sum(area_of[n] for n in wfd_names)
area_of["Total mini-surveys"] = sum(area_of[n] for n in mini_names)
pd.Series(area_of).round(1).to_frame("Area (deg2)")

## 2. Load the visits and attribute each visit to one category

All visits are read once and sorted in time. Time is expressed in **years since the survey start** (`SURVEY_START_MJD`).

**Micro-surveys** are selected from `scheduler_note`: `ToO`, `twilight_near_sun`, `RGES`, and `DD` (DDF, RGES excluded as in `02_SurveyAreas` and `04_DDF_NVisitsVsTime`). A visit has at most one micro-survey tag (priority in the order above).

**Other visits** get the label of the HEALPix pixel (nside 64) containing the visit centre (`fieldRA`, `fieldDec`).

In [ ]:
query = (
    "select observationStartMJD, night, band, fieldRA, fieldDec, "
    "visitExposureTime, scheduler_note from observations"
)
conn = sqlite3.connect(opsdb)
visits = pd.read_sql(query, conn)
conn.close()

visits = visits.sort_values("observationStartMJD").reset_index(drop=True)
visits["years"] = (visits["observationStartMJD"] - SURVEY_START_MJD) / YEAR
N_ALL = len(visits)
print(f'{N_ALL} visits in the survey, duration {visits["years"].max():.2f} years')

# Micro-surveys, from scheduler_note (one tag per visit, priority ToO > TwilightNEO > RGES > DDF)
note = visits["scheduler_note"].fillna("")
is_too = note.str.contains("ToO").values
is_twilight = note.str.contains("twilight_near_sun").values
is_rges = note.str.contains("RGES").values
is_ddf = note.str.contains("DD").values & ~is_rges
visits["micro"] = np.select(
    [is_too, is_twilight, is_rges, is_ddf],
    ["ToO", "TwilightNEO", "RGES", "DDF"],
    default="",
)
is_micro = (visits["micro"] != "").values

# Third visits of triplets: WFD visits, kept only as an overlay in the micro-survey figure
is_triplet = (note.str.contains("long") & ~note.str.contains("blob")).values

# Label of the HEALPix pixel containing the visit centre (RA, Dec in degrees)
pix = hp.ang2pix(NSIDE, visits["fieldRA"].values, visits["fieldDec"].values, lonlat=True)
visits["label"] = labels[pix]

print(visits["micro"].replace("", "not micro").value_counts())

**Boolean masks.** `masks` stores, for each region and each total, a boolean array over the time-sorted visits (True when the visit belongs to it). Check: the three totals plus `Unassigned` must add up to the number of visits of the survey.

In [ ]:
sky_label = visits["label"].values
not_micro = ~is_micro

masks = {}
group_type = {}

# WFD regions and total WFD
for name, labs in WFD_REGIONS.items():
    masks[name] = not_micro & np.isin(sky_label, labs)
    group_type[name] = "WFD"
masks["Total WFD"] = np.any([masks[n] for n in wfd_names], axis=0)
group_type["Total WFD"] = "WFD"

# Mini-surveys and total mini-surveys
for name, labs in MINI_SURVEYS.items():
    masks[name] = not_micro & np.isin(sky_label, labs)
    group_type[name] = "mini-survey"
masks["Total mini-surveys"] = np.any([masks[n] for n in mini_names], axis=0)
group_type["Total mini-surveys"] = "mini-survey"

# Micro-surveys and total micro-surveys
for name in micro_names:
    masks[name] = visits["micro"].values == name
    group_type[name] = "micro-survey"
masks["Total micro-surveys"] = is_micro
group_type["Total micro-surveys"] = "micro-survey"

# Visits that are not micro-survey visits and fall in a pixel without WFD or mini-survey label
masks["Unassigned"] = not_micro & ~masks["Total WFD"] & ~masks["Total mini-surveys"]
group_type["Unassigned"] = "check"
masks["All visits"] = np.ones(N_ALL, dtype=bool)
group_type["All visits"] = "all"

n_sum = sum(masks[n].sum() for n in ["Total WFD", "Total mini-surveys", "Total micro-surveys", "Unassigned"])
print(f"WFD + mini + micro + unassigned = {n_sum}, all visits = {N_ALL}")
assert n_sum == N_ALL, "a visit is counted twice or not at all"
print(f'Unassigned visits: {masks["Unassigned"].sum()} ({100 * masks["Unassigned"].sum() / N_ALL:.3f}%)')
print(f"Third visits of triplets (inside the WFD or mini-surveys): {is_triplet.sum()}")

## 3. Summary table

For each region: number of visits, share of the whole survey, number of distinct nights, time of the first and last visit (years since survey start), sky area of the region (deg2, from the target map; not defined for the micro-surveys), total exposure time (hours) and number of visits per band. The table is saved to `DATA07_WFDMiniMicroS_NVisitsVsTime/`.

The second table gives the **number of visits in each survey year** (Y1 = first year) for each region.

In [ ]:
def summary_row(mask, area):
    d = visits[mask]
    row = {
        "N visits": len(d),
        "% of survey": 100 * len(d) / N_ALL,
        "N nights": d["night"].nunique(),
        "first (yr)": d["years"].min(),
        "last (yr)": d["years"].max(),
        "area (deg2)": area,
        "exposure (h)": d["visitExposureTime"].sum() / 3600,
    }
    for b in BANDS:
        row[b] = int((d["band"] == b).sum())
    return pd.Series(row)


summary = pd.DataFrame({n: summary_row(m, area_of.get(n, np.nan)) for n, m in masks.items()}).T
summary.insert(0, "type", pd.Series(group_type))
summary["N visits"] = summary["N visits"].astype(int)
summary["N nights"] = summary["N nights"].astype(int)
for b in BANDS:
    summary[b] = summary[b].astype(int)
for col in ["% of survey", "first (yr)", "last (yr)", "area (deg2)", "exposure (h)"]:
    summary[col] = summary[col].astype(float).round(2)

summary.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_region_summary.csv"))
summary

In [ ]:
# Number of visits in each survey year, for each region
t_all = visits["years"].values  # sorted in time
year_edges = np.arange(0, int(np.ceil(t_all.max())) + 1)
per_year = pd.DataFrame(
    {n: np.diff(np.searchsorted(t_all[m], year_edges, side="right")) for n, m in masks.items()},
    index=[f"Y{i + 1}" for i in range(len(year_edges) - 1)],
).T
per_year.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_visits_per_year.csv"))
per_year

## 4. Cumulative visits versus time

The cumulative number of visits of each region is computed on a regular time grid of 3000 points (about 1 day) and saved to `DATA07_WFDMiniMicroS_NVisitsVsTime/` (`..._cumulative_visits_vs_time.csv`). Each figure has two panels with the same curves: **linear scale** (left) and **log scale** (right), because a region with few visits is invisible next to the total on a linear scale. The thick black curve is the total of the figure. The number of visits of each region at the end of the survey is given in the legend.

Flat parts of a curve are periods without visits for this region (not visible, or not scheduled); a steeper slope means a higher visit rate. A region observed at a constant rate gives a straight line.

In [ ]:
# One colour per region (same palette for WFD regions, mini-surveys and micro-surveys, one group per figure)
tab = plt.get_cmap("tab10")
colors = {}
for group in (wfd_names, mini_names, micro_names):
    colors.update({n: tab(i % 10) for i, n in enumerate(group)})
# The sum of all the DDFs gets its own colour
colors["DDF"] = "C4"
# Colours of the totals, used in the overview panels
colors.update({"Total WFD": "C0", "Total mini-surveys": "C1", "Total micro-surveys": "C2"})

# Cumulative number of visits on a regular time grid
grid = np.linspace(0.0, t_all.max(), 3000)
cum = pd.DataFrame({"years": grid})
for n, m in masks.items():
    cum[n] = np.searchsorted(t_all[m], grid, side="right")
cum["Triplet (third)"] = np.searchsorted(t_all[is_triplet], grid, side="right")
cum.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_cumulative_visits_vs_time.csv"), index=False)


def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=150)


def plot_cumulative(names, total, title, stem, with_triplets=False):
    # Cumulative visits versus time: linear scale (left) and log scale (right)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    for ax, logy in zip(axes, (False, True)):
        for n in names:
            if masks[n].sum() == 0:
                continue
            ax.plot(cum["years"], cum[n], color=colors[n], lw=1.8, label=f"{n} ({masks[n].sum()})")
        if with_triplets and is_triplet.sum() > 0:
            ax.plot(
                cum["years"],
                cum["Triplet (third)"],
                color="gray",
                lw=1.5,
                ls="--",
                label=f"Triplet third visits, in WFD ({is_triplet.sum()})",
            )
        ax.plot(cum["years"], cum[total], color="k", lw=2.8, label=f"{total} ({masks[total].sum()})")
        ax.set_xlabel("Time since survey start (years)")
        ax.set_ylabel("Cumulative number of visits")
        ax.set_xlim(left=0)
        ax.grid(alpha=0.3)
        if logy:
            ax.set_yscale("log")
            ax.set_ylim(bottom=1)
            ax.set_title("log scale")
        else:
            ax.set_ylim(bottom=0)
            ax.set_title("linear scale")
        ax.legend(title="region (total visits)")
    fig.suptitle(f"{title} - {opsim_name}")
    fig.tight_layout()
    save_fig(fig, stem)
    return fig

### Figure 1 - Cumulative visits in the WFD regions (`<run>_wfd_cumulative.png`)

One curve per WFD region (low dust, bulge, Virgo, SMC-LMC) and the total WFD (thick black). It shows when each region is observed and how the rolling cadence and the seasonal visibility shape the accumulation of visits.

In [ ]:
fig1 = plot_cumulative(wfd_names, "Total WFD", "Cumulative visits in the WFD regions", "wfd_cumulative")

### Figure 2 - Cumulative visits in the mini-surveys (`<run>_mini_cumulative.png`)

One curve per mini-survey (NES, SCP, MW non-WFD, and any other label found in the target map) and the total of the mini-surveys (thick black).

In [ ]:
fig2 = plot_cumulative(
    mini_names, "Total mini-surveys", "Cumulative visits in the mini-surveys", "mini_cumulative"
)

### Figure 3 - Cumulative visits in the micro-surveys (`<run>_micro_cumulative.png`)

One curve per micro-survey (DDF, RGES, near-sun twilight, ToO; a micro-survey without any visit in the run is not drawn) and the total of the micro-surveys (thick black). The grey dashed curve gives the third visits of triplets for comparison: they belong to the WFD and are **not** included in the micro-survey total. The DDF curve is the sum of all DDFs, whose detail is in `04_DDF_NVisitsVsTime`.

In [ ]:
fig3 = plot_cumulative(
    micro_names,
    "Total micro-surveys",
    "Cumulative visits in the micro-surveys",
    "micro_cumulative",
    with_triplets=True,
)

### Figure 4 - Overview: WFD, mini-surveys and micro-surveys (`<run>_overview_cumulative.png`)

The three totals (total WFD, total mini-surveys, total micro-surveys) compared with the number of all visits of the survey (thick black).

In [ ]:
fig4 = plot_cumulative(
    ["Total WFD", "Total mini-surveys", "Total micro-surveys"],
    "All visits",
    "Cumulative visits: WFD, mini-surveys, micro-surveys",
    "overview_cumulative",
)

## 5. Fraction of the survey visits

The fraction of a region is the number of visits of the region accumulated so far divided by the number of all visits accumulated so far, in percent. Each figure has four panels: WFD regions, mini-surveys, micro-surveys (each with its total in thick black), and the three totals together with the sum of all the DDFs (`All DDF`, also included in the total micro-surveys). The sum of all the DDFs is also drawn as a dashed curve in the WFD and mini-survey panels, for comparison. The last point of each curve is the share given by the column `% of survey` of the summary table.

### Figure 5 - Fraction versus the total number of visits (`<run>_fraction_vs_nvisits.png`)

The horizontal axis is the **cumulative number of visits of the whole survey** (in millions), counted in time order, so the curve answers: among the first N visits of the survey, which fraction went to each region? The top axis gives the corresponding time since survey start. The curves start at the 5000th visit, before which the fractions are dominated by statistical fluctuations. The table of the curves is saved to `DATA07_WFDMiniMicroS_NVisitsVsTime/` (`..._cumulative_visits_vs_nvisits.csv`).

In [ ]:
# Number of visits of each region among the first N visits of the survey (visits sorted in time)
n_grid = np.unique(np.linspace(5000, N_ALL, 1500).astype(int))
cum_n = pd.DataFrame({"n_visits": n_grid})
for n, m in masks.items():
    cum_n[n] = np.cumsum(m)[n_grid - 1]
cum_n["Triplet (third)"] = np.cumsum(is_triplet)[n_grid - 1]
cum_n.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_cumulative_visits_vs_nvisits.csv"), index=False)

# Conversion between the total number of visits (millions) and the time (years), for the top axis
k_all = np.arange(1, N_ALL + 1)


def millions_to_years(x):
    return np.interp(np.asarray(x) * 1e6, k_all, t_all)


def years_to_millions(t):
    return np.interp(np.asarray(t), t_all, k_all) / 1e6


# Legend text of the curves: DDF is the sum of all the DDFs, not a single DDF
LABEL = {"DDF": "All DDF (sum of all DDFs)"}

PANELS = [
    ("WFD regions", wfd_names, "Total WFD"),
    ("Mini-surveys", mini_names, "Total mini-surveys"),
    ("Micro-surveys", micro_names, "Total micro-surveys"),
    ("Totals", ["Total WFD", "Total mini-surveys", "Total micro-surveys", "DDF"], None),
]


def plot_fraction(df, xcol, xlabel, title, stem, xscale=1.0, top_axis_years=False):
    # Fraction (%) of the cumulative visits of each region over the cumulative visits of the survey
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    x = df[xcol].values / xscale
    denom = np.maximum(df["All visits"].values, 1)
    for ax, (panel, names, total) in zip(axes.ravel(), PANELS):
        for n in names:
            if masks[n].sum() == 0:
                continue
            ax.plot(x, 100 * df[n].values / denom, color=colors[n], lw=1.8, label=LABEL.get(n, n))
        # The sum of all the DDFs is drawn in every panel as a reference (dashed when not part of the panel)
        if "DDF" not in names and masks["DDF"].sum() > 0:
            ax.plot(
                x, 100 * df["DDF"].values / denom, color=colors["DDF"], lw=1.8, ls="--", label=LABEL["DDF"]
            )
        if total is not None:
            ax.plot(x, 100 * df[total].values / denom, color="k", lw=2.8, label=total)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Fraction of cumulative visits (%)")
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
        ax.set_title(panel, pad=40 if top_axis_years else 6)
        ax.legend()
        if top_axis_years:
            sec = ax.secondary_xaxis("top", functions=(millions_to_years, years_to_millions))
            sec.set_xlabel("Time since survey start (years)")
    fig.suptitle(f"{title} - {opsim_name}")
    fig.tight_layout()
    save_fig(fig, stem)
    return fig


fig5 = plot_fraction(
    cum_n,
    "n_visits",
    "Cumulative number of visits of the whole survey (millions)",
    "Fraction of the survey visits versus total number of visits",
    "fraction_vs_nvisits",
    xscale=1e6,
    top_axis_years=True,
)

### Figure 6 - Fraction versus time (`<run>_fraction_vs_time.png`)

Same four panels as Figure 5, but the horizontal axis is the **time since survey start** (years), as in Figure 6 of `04_DDF_NVisitsVsTime` for the DDFs. The curves start at 0.05 year.

In [ ]:
fig6 = plot_fraction(
    cum[cum["years"] > 0.05],
    "years",
    "Time since survey start (years)",
    "Fraction of the survey visits versus time",
    "fraction_vs_time",
)

## 6. Summary: area and visits of the main survey regions

One row per main region: `DDF` (all DDFs together), `WFD` (sum of the four WFD regions), `Mini-surveys` (sum of the mini-surveys), `Micro-surveys (without DDF)` (RGES, near-sun twilight and ToO; the DDFs have their own row) and `Others (unassigned visits)` (visits in a pixel without label). The categories are exclusive for the visits, so the visit fractions add up to 100%.

- **Area**: for WFD and mini-surveys, area of their pixels in the scheduler target map. DDFs and micro-surveys are not in the target map: their area is the sky area covered by the field of view (disc of radius 1.75 deg) of their visits, estimated on a HEALPix grid (nside 256 for the DDFs, 64 for the micro-surveys). The areas of different rows overlap (the micro-surveys observe sky that is also in the WFD), so they are **not additive**: the table gives the fraction of the whole sky (41253 deg2) and the fraction of the sum of the listed areas, and the `Total` row counts overlaps twice.
- **Visits**: number of visits during the first 10 years of the survey and fraction of all the visits of these 10 years.

The table is saved to `DATA07_WFDMiniMicroS_NVisitsVsTime/` (`..._main_regions_summary_10yr.csv`).

In [ ]:
# Sky area covered by the visits of a category: union of discs of radius FOV_RADIUS (field of view)
# around the HEALPix pixels containing the visit centres
FOV_RADIUS = 1.75  # deg, one pointing covers about 9.6 deg2


def covered_area(mask, nside):
    if mask.sum() == 0:
        return 0.0
    ra = visits["fieldRA"].values[mask]
    dec = visits["fieldDec"].values[mask]
    pix_centres = np.unique(hp.ang2pix(nside, ra, dec, lonlat=True))
    covered = np.zeros(hp.nside2npix(nside), dtype=bool)
    for p in pix_centres:
        covered[hp.query_disc(nside, hp.pix2vec(nside, p), np.radians(FOV_RADIUS))] = True
    return covered.sum() * hp.nside2pixarea(nside, degrees=True)


# Micro-surveys other than the DDFs (RGES, near-sun twilight, ToO)
mask_micro_other = is_micro & ~masks["DDF"]
area_ddf = covered_area(masks["DDF"], 256)
area_micro_other = covered_area(mask_micro_other, 64)
print(f"Area covered by the DDFs: {area_ddf:.1f} deg2")
print(f"Area covered by the other micro-surveys: {area_micro_other:.1f} deg2")

In [ ]:
FULL_SKY = 4 * np.pi * (180 / np.pi) ** 2  # deg2

within = t_all <= 10.0  # first 10 years of the survey
n_10yr = int(within.sum())

# category name -> (mask of its visits, area in deg2)
categories = {
    "DDF": (masks["DDF"], area_ddf),
    "WFD": (masks["Total WFD"], area_of["Total WFD"]),
    "Mini-surveys": (masks["Total mini-surveys"], area_of["Total mini-surveys"]),
    "Micro-surveys (without DDF)": (mask_micro_other, area_micro_other),
    "Others (unassigned visits)": (masks["Unassigned"], np.nan),
}

rows = {}
for name, (m, area) in categories.items():
    n = int((m & within).sum())
    rows[name] = {
        "Area (deg2)": area,
        "Area (% of sky)": 100 * area / FULL_SKY,
        "Area (% of summed areas)": np.nan,
        "N visits (10 yr)": n,
        "Visits (% of all, 10 yr)": 100 * n / n_10yr,
    }
table = pd.DataFrame(rows).T.astype(float)

# Fraction of the summed areas, and total row (overlapping areas are counted twice)
total_area = table["Area (deg2)"].sum()
table["Area (% of summed areas)"] = 100 * table["Area (deg2)"] / total_area
table.loc["Total (areas summed)"] = [
    total_area,
    100 * total_area / FULL_SKY,
    100.0,
    table["N visits (10 yr)"].sum(),
    table["Visits (% of all, 10 yr)"].sum(),
]
assert int(table.loc["Total (areas summed)", "N visits (10 yr)"]) == n_10yr, "categories are not exclusive"
table["N visits (10 yr)"] = table["N visits (10 yr)"].astype(int)
table = table.round(2)

table.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_main_regions_summary_10yr.csv"))
table

In [ ]:
table.drop(
    columns=["Area (% of summed areas)", "Area (% of sky)"],
    index=["Others (unassigned visits)"],
    inplace=False,
)

In [ ]:
table_clean = table.drop(
    columns=["Area (% of summed areas)", "Area (% of sky)"],
    index=["Others (unassigned visits)"],
    errors="ignore",
)

table_clean.style.format(
    {"Area (deg2)": "{:,.0f}", "N visits (10 yr)": "{:,.0f}", "Visits (% of all, 10 yr)": "{:.1f}%"}
).set_properties(
    subset=["Area (deg2)", "N visits (10 yr)", "Visits (% of all, 10 yr)"], **{"text-align": "right"}
).set_table_styles(
    [
        {
            "selector": "th",
            "props": [
                ("background-color", "grey"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "8px"),
            ],
        },
        {"selector": "td", "props": [("border", "1px solid #ddd")]},
        {
            "selector": "table",
            "props": [("border-collapse", "collapse"), ("width", "100%"), ("font-size", "13px")],
        },
    ]
)